In [2]:
import os
import numpy as np
import cv2
import joblib
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
import gc
import matplotlib.pyplot as plt
import uuid
from imgaug import augmenters as iaa
from tqdm import tqdm
import time

In [3]:
img_size = (64, 64)
data_dir = 'split_dataset'
original_train_dir = os.path.join(data_dir, 'train')
augmented_train_dir = os.path.join(data_dir, 'train_augmented')
val_dir = os.path.join(data_dir, 'val')
test_dir = os.path.join(data_dir, 'test')
TARGET_PER_CLASS = 800
n_splits = 5

In [4]:
augmenter = iaa.Sequential([
    iaa.Affine(
        rotate=(-10, 10),
        shear=(-15, 15),
        scale=(0.85, 1.15),
        translate_percent={"x": (-0.2, 0.2), "y": (-0.2, 0.2)}
    ),
    iaa.Multiply((0.85, 1.15)),
    iaa.Fliplr(0.0),
    iaa.GaussianBlur(sigma=(0, 1.0))
])

def preprocess_image(image, target_size=(64, 64)):
    image = cv2.resize(image, target_size)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.astype("float32") / 255.0
    return image

def load_images_from_folder(folder):
    X, y = [], []
    for class_name in os.listdir(folder):
        class_path = os.path.join(folder, class_name)
        if not os.path.isdir(class_path):
            continue
        for file in os.listdir(class_path):
            file_path = os.path.join(class_path, file)
            image = cv2.imread(file_path)
            if image is None:
                continue
            image = preprocess_image(image, target_size=img_size)
            X.append(image)
            y.append(class_name)
    return np.array(X), np.array(y)

def augment_train_set():
    os.makedirs(augmented_train_dir, exist_ok=True)
    for label in os.listdir(original_train_dir):
        orig_path = os.path.join(original_train_dir, label)
        aug_path = os.path.join(augmented_train_dir, label)
        os.makedirs(aug_path, exist_ok=True)

        images = [cv2.imread(os.path.join(orig_path, img)) for img in os.listdir(orig_path) if img.lower().endswith(('.jpg', '.png', '.jpeg'))]
        images = [img for img in images if img is not None]
        current_count = len(images)

        if current_count >= TARGET_PER_CLASS:
            print(f"[✓] {label}: {current_count} images — no augmentation needed.")
            continue

        needed = TARGET_PER_CLASS - current_count
        print(f"[•] Augmenting {label}: {needed} images")

        for i in tqdm(range(needed), desc=f"Augmenting {label}"):
            img = images[i % len(images)]
            aug_img = augmenter(image=img)
            filename = f"{uuid.uuid4().hex}.jpg"
            cv2.imwrite(os.path.join(aug_path, filename), aug_img)

    print("Augmentation complete.\n")

In [6]:
augment_train_set()
X_train, y_train = load_images_from_folder(original_train_dir)
X_aug, y_aug = load_images_from_folder(augmented_train_dir)
X_val, y_val = load_images_from_folder(val_dir)
X_test, y_test = load_images_from_folder(test_dir)

X_all = np.concatenate([X_train, X_aug, X_val])
y_all = np.concatenate([y_train, y_aug, y_val])
X_all_flat = X_all.reshape(len(X_all), -1).astype(np.float32)

le = LabelEncoder()
y_all_enc = le.fit_transform(y_all)

print(f"Total samples: {len(X_all)}")

[•] Augmenting प: 172 images


Augmenting प: 100%|██████████| 172/172 [00:00<00:00, 247.98it/s]


[•] Augmenting श: 172 images


Augmenting श: 100%|██████████| 172/172 [00:00<00:00, 250.66it/s]


[•] Augmenting ढ: 172 images


...

[•] Augmenting ख: 172 images


Augmenting ख: 100%|██████████| 172/172 [00:00<00:00, 245.07it/s]


[•] Augmenting व: 172 images


Augmenting व: 100%|██████████| 172/172 [00:00<00:00, 249.41it/s]


Augmentation complete.

Total samples: 35440


In [ ]:
print("\nStarting Stratified K-Fold Cross-Validation with SVM...\n")
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
all_y_true, all_y_pred = [], []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_all_flat, y_all_enc), 1):
    print(f"--- Fold {fold} ---")
    
    X_train_fold = X_all_flat[train_idx]
    X_test_fold = X_all_flat[test_idx]
    y_train_fold = y_all_enc[train_idx]
    y_test_fold = y_all_enc[test_idx]

    pca = PCA(n_components=0.95)
    X_train_pca = pca.fit_transform(X_train_fold)
    X_test_pca = pca.transform(X_test_fold)

    svm = SVC(kernel='rbf', probability=True, random_state=42)
    svm.fit(X_train_pca, y_train_fold)
    
    y_pred = svm.predict(X_test_pca)
    all_y_true.extend(y_test_fold)
    all_y_pred.extend(y_pred)

    print(classification_report(y_test_fold, y_pred, target_names=le.classes_))
    
    del X_train_fold, X_test_fold, X_train_pca, X_test_pca, svm, pca
    gc.collect()
    time.sleep(0.5)


Starting Stratified K-Fold Cross-Validation with SVM...

--- Fold 1 ---


In [ ]:
print("\nTraining final model on full dataset...")
final_pca = PCA(n_components=0.95)
X_all_pca = final_pca.fit_transform(X_all_flat)

final_svm = SVC(kernel='rbf', probability=True, random_state=42)
final_svm.fit(X_all_pca, y_all_enc)

In [ ]:
print("\nGenerating Confusion Matrix...")
cm = confusion_matrix(all_y_true, all_y_pred)
fig, ax = plt.subplots(figsize=(12, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax)
plt.title("Cross-Validation Confusion Matrix")
plt.show()

In [ ]:
train_acc = final_svm.score(X_all_pca, y_all_enc)
print(f"\nTraining-set accuracy: {train_acc:.4f}")


In [ ]:
joblib.dump(final_svm, 'final_svm_model.pkl')
joblib.dump(final_pca, 'final_svm_pca_model.pkl')
joblib.dump(le, 'svm_label_encoder.pkl')
print("\n SVM, PCA, and LabelEncoder models saved.")